# 직접 해 보기 2 — 파이썬 코드로 따라 해 보기

AI가 내놓은 결과만 보고 그것이 어떻게 나왔는지 모르면, 그 결과를 믿을 근거도 없다. AI도 실수를 하므로 결과는 점검해야 하고, 점검하려면 결과를 만든 파이썬 코드를 읽을 줄 알아야 한다.

이 노트북은 이 장에서 다룬 계산을 파이썬으로 직접 해 본다. 척도 점수와 크론바흐 α는 평균과 분산만으로 계산하고, 요인분석은 `sklearn`의 `FactorAnalysis` 하나로 한다. 모든 계산을 다섯 성격에 똑같이 하므로, 외향성 말고 다른 척도의 결과도 함께 볼 수 있다.

1. 데이터 읽기 — 원 사이트에서 바로
2. 척도와 채점 키 — 역문항의 답 뒤집기와 척도 점수(표 2-1, 그림 2-2)
3. 크론바흐 α — 보고서의 표, 뒤집기 전과 뒤, 문항 수, 문항 하나를 빼면(표 3-2, 그림 3-3, 표 3-3)
4. 요인분석 — 보고서의 그림, 문항마다의 적재값, 고유값, 회전(표 4-2, 표 4-3, 그림 4-4)
5. 감쇠 — 문항 하나로 측정하면(그림 5-1, 표 5-1)
6. 더 해 보기: 채점 키를 자료에서 짐작하면
7. 더 해 보기: 하만 단일요인검정

## 준비 — 패키지와 글꼴

`import`로 쓸 패키지를 불러온다. `as` 뒤의 이름은 줄임말이고, 이후 코드에서는 `pd`, `smf`처럼 줄임말로 부른다. 이 장에서 쓰는 패키지는 모두 Anaconda와 Colab에 기본으로 들어 있다.

matplotlib는 기본 글꼴에 한글이 없어서 그림의 한글이 네모로 깨진다. 그래서 한글 글꼴을 지정한다. 윈도우는 맑은 고딕(`Malgun Gothic`), 맥은 애플고딕(`AppleGothic`)을 쓴다. Colab에는 한글 글꼴이 없으므로 새 셀에서 `!apt-get -qq install fonts-nanum`을 실행하고, 런타임을 다시 시작한 뒤 `NanumGothic`을 지정한다.

In [ ]:
import io, os, zipfile, urllib.request             # 파일 확인, 원 사이트에서 압축 파일 받기
import numpy as np                                 # 무작위 뽑기, 고유값
import pandas as pd                                # 표(데이터프레임) 다루기
import statsmodels.formula.api as smf              # 회귀분석
from sklearn.decomposition import FactorAnalysis   # 요인분석
import matplotlib.pyplot as plt                    # 그림

# 그림의 한글 글꼴
# 맥은 'AppleGothic', Colab은 'NanumGothic'(설치 필요)
plt.rcParams['font.family'] = 'Malgun Gothic'   # 윈도우
plt.rcParams['axes.unicode_minus'] = False      # 음수 부호(-) 깨짐 방지

## 1. 데이터 읽기 — 원 사이트에서 바로

이 장의 자료는 Open Psychometrics Project가 공개한 성격 검사 원자료다(2012년 무렵 수집). 재배포를 허락하는 이용 조건이 없어서 책 사이트에는 파일을 두지 않았고, 이 노트북이 원 사이트에서 바로 읽는다(인터넷에 연결돼 있어야 한다). BIG5.zip이라는 압축 파일 안에 `data.csv`(답)와 `codebook.txt`(문항의 문장)가 들어 있다.

`urllib.request`로 압축 파일을 받고, `zipfile`로 그 안의 `data.csv`를 연다. 사이트가 브라우저가 아닌 접속을 막는 경우가 있어서 `User-Agent`에 브라우저 이름을 적어 보낸다. `data.csv`는 이름과 달리 값이 탭으로 나뉘어 있으므로 `sep='\t'`를 준다. 사이트에 접속되지 않으면(접속 오류가 나면) 브라우저로 BIG5.zip을 받아 풀고, 그 안의 `data.csv`를 이 노트북과 같은 폴더에 둔 뒤 다시 실행한다. 코드는 같은 폴더에 `data.csv`가 있으면 그 파일을 먼저 읽는다(`os.path.exists`가 파일이 있는지 확인한다).

| 변수 | 뜻 |
|---|---|
| `E1` ~ `E10`, `N1` ~ `N10`, `A1` ~ `A10`, `C1` ~ `C10`, `O1` ~ `O10` | 50문항의 답(1 = 동의하지 않음 ~ 5 = 동의, 0 = 답하지 않음). 앞 글자가 성격 |
| `age`, `gender`, `country` 등 7개 | 나이, 성별(1 남성, 2 여성, 3 기타), 접속한 나라 등 |

답하지 않은 문항은 0으로 적혀 있다. 0을 그대로 두고 평균하면 그 사람의 점수가 낮아지므로, 50문항 모두 답한 사람만 남긴다. `(raw[ITEMS] > 0).all(axis=1)`은 50문항이 모두 0보다 큰 사람이면 참이다.

In [ ]:
if os.path.exists('data.csv'):                            # 직접 받아 푼 파일이 있으면 그것을
    raw = pd.read_csv('data.csv', sep='\t')
else:                                                     # 없으면 원 사이트에서 바로
    URL = 'https://openpsychometrics.org/_rawdata/BIG5.zip'
    req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(req).read()))
    raw = pd.read_csv(z.open('BIG5/data.csv'), sep='\t')

TRAITS = {'E': '외향성', 'N': '신경성', 'A': '친화성', 'C': '성실성', 'O': '개방성'}
ITEMS = [t + str(i) for t in TRAITS for i in range(1, 11)]   # E1, ..., O10
print('응답자', len(raw), '명')
raw = raw[(raw[ITEMS] > 0).all(axis=1)].reset_index(drop=True)
print('50문항 모두 답한 사람', len(raw), '명')
print('나라 상위 3:', raw['country'].value_counts().head(3).to_dict())
raw.head()

## 2. 척도와 채점 키 — 역문항의 답 뒤집기와 척도 점수

먼저 문항 50개의 문장을 표로 만든다. 문장은 이 책이 우리말로 옮긴 것이고, 영어 원문(자료와 함께 받는 `codebook.txt`)을 옆에 나란히 둔다. `REV`는 IPIP 채점 키의 역문항 18개, 즉 문장이 그 성격을 거꾸로 묻는 문항이다. 어느 문항이 역문항인지는 자료의 숫자에 적혀 있지 않으므로, 문장을 보고 정한 목록을 직접 적는다(2절). 표에서 문장과 역문항 표시를 함께 읽어 보자.

In [ ]:
ITEM_KO = {
    'E1': '나는 모임의 분위기를 띄우는 사람이다.', 'E2': '나는 말이 많지 않다.',
    'E3': '나는 사람들 사이에서 편안하다.', 'E4': '나는 뒤로 물러나 있는 편이다.',
    'E5': '나는 먼저 말을 건다.', 'E6': '나는 할 말이 별로 없다.',
    'E7': '나는 모임에서 여러 사람과 이야기한다.', 'E8': '나는 주목받는 것을 좋아하지 않는다.',
    'E9': '나는 관심의 한가운데에 있어도 괜찮다.', 'E10': '나는 낯선 사람 앞에서 조용하다.',
    'N1': '나는 쉽게 스트레스를 받는다.', 'N2': '나는 대개 느긋하다.',
    'N3': '나는 이런저런 걱정을 한다.', 'N4': '나는 기분이 가라앉는 일이 드물다.',
    'N5': '나는 쉽게 동요한다.', 'N6': '나는 쉽게 속상해한다.',
    'N7': '나는 기분이 자주 바뀐다.', 'N8': '나는 기분이 자주 오락가락한다.',
    'N9': '나는 쉽게 짜증이 난다.', 'N10': '나는 기분이 가라앉을 때가 많다.',
    'A1': '나는 다른 사람을 별로 걱정하지 않는다.', 'A2': '나는 사람들에게 관심이 있다.',
    'A3': '나는 사람들을 모욕한다.', 'A4': '나는 다른 사람의 감정에 공감한다.',
    'A5': '나는 다른 사람의 문제에 관심이 없다.', 'A6': '나는 마음이 여리다.',
    'A7': '나는 다른 사람에게 그다지 관심이 없다.', 'A8': '나는 다른 사람을 위해 시간을 낸다.',
    'A9': '나는 다른 사람의 감정을 느낀다.', 'A10': '나는 사람들을 편안하게 해 준다.',
    'C1': '나는 늘 준비가 되어 있다.', 'C2': '나는 물건을 아무 데나 둔다.',
    'C3': '나는 세세한 것까지 신경 쓴다.', 'C4': '나는 일을 엉망으로 만든다.',
    'C5': '나는 해야 할 일을 바로 해치운다.', 'C6': '나는 물건을 제자리에 두는 것을 자주 잊는다.',
    'C7': '나는 정돈된 것을 좋아한다.', 'C8': '나는 맡은 일을 게을리한다.',
    'C9': '나는 일정을 지킨다.', 'C10': '나는 일을 빈틈없이 한다.',
    'O1': '나는 어휘가 풍부하다.', 'O2': '나는 추상적인 생각을 이해하기 어렵다.',
    'O3': '나는 상상력이 풍부하다.', 'O4': '나는 추상적인 생각에 관심이 없다.',
    'O5': '나는 좋은 아이디어를 낸다.', 'O6': '나는 상상력이 좋지 않다.',
    'O7': '나는 이해가 빠르다.', 'O8': '나는 어려운 낱말을 쓴다.',
    'O9': '나는 이것저것 곰곰이 생각한다.', 'O10': '나는 아이디어가 넘친다.'}
ITEM_EN = {                                          # 원문(codebook.txt)
    'E1': "I am the life of the party.", 'E2': "I don't talk a lot.",
    'E3': "I feel comfortable around people.", 'E4': "I keep in the background.",
    'E5': "I start conversations.", 'E6': "I have little to say.",
    'E7': "I talk to a lot of different people at parties.", 'E8': "I don't like to draw attention to myself.",
    'E9': "I don't mind being the center of attention.", 'E10': "I am quiet around strangers.",
    'N1': "I get stressed out easily.", 'N2': "I am relaxed most of the time.",
    'N3': "I worry about things.", 'N4': "I seldom feel blue.",
    'N5': "I am easily disturbed.", 'N6': "I get upset easily.",
    'N7': "I change my mood a lot.", 'N8': "I have frequent mood swings.",
    'N9': "I get irritated easily.", 'N10': "I often feel blue.",
    'A1': "I feel little concern for others.", 'A2': "I am interested in people.",
    'A3': "I insult people.", 'A4': "I sympathize with others' feelings.",
    'A5': "I am not interested in other people's problems.", 'A6': "I have a soft heart.",
    'A7': "I am not really interested in others.", 'A8': "I take time out for others.",
    'A9': "I feel others' emotions.", 'A10': "I make people feel at ease.",
    'C1': "I am always prepared.", 'C2': "I leave my belongings around.",
    'C3': "I pay attention to details.", 'C4': "I make a mess of things.",
    'C5': "I get chores done right away.", 'C6': "I often forget to put things back in their proper place.",
    'C7': "I like order.", 'C8': "I shirk my duties.",
    'C9': "I follow a schedule.", 'C10': "I am exacting in my work.",
    'O1': "I have a rich vocabulary.", 'O2': "I have difficulty understanding abstract ideas.",
    'O3': "I have a vivid imagination.", 'O4': "I am not interested in abstract ideas.",
    'O5': "I have excellent ideas.", 'O6': "I do not have a good imagination.",
    'O7': "I am quick to understand things.", 'O8': "I use difficult words.",
    'O9': "I spend time reflecting on things.", 'O10': "I am full of ideas."}
REV = ['E2', 'E4', 'E6', 'E8', 'E10', 'N2', 'N4', 'A1', 'A3', 'A5', 'A7',
       'C2', 'C4', 'C6', 'C8', 'O2', 'O4', 'O6']     # 채점 키: 역문항
MAX = 5                                              # 눈금의 가장 큰 값(5점 척도)

def cols(t):
    return [t + str(i) for i in range(1, 11)]       # 성격 t의 문항 10개

book = pd.DataFrame({'성격': [TRAITS[c[0]] for c in ITEMS], '문장': [ITEM_KO[c] for c in ITEMS],
                     '원문': [ITEM_EN[c] for c in ITEMS],
                     '역문항': ['역' if c in REV else '' for c in ITEMS]}, index=ITEMS)
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_colwidth', 80)              # 긴 문장이 잘리지 않게
book

2절의 표 2-1을 만든다. 응답자를 E1에 준 답으로 다섯 그룹으로 나누고, 그룹마다 E3(정문항)과 E2(역문항)에 준 답의 평균을 낸다. `groupby('E1')`이 E1의 답마다 사람을 묶는다. E1에 높게 답한 그룹일수록 E3의 평균은 높아지고 E2의 평균은 낮아지는 것을 확인한다.

그다음 역문항의 답을 뒤집는다. 뒤집기는 눈금의 가장 작은 값과 가장 큰 값을 더한 수에서 답을 빼는 것이다(1 ~ 5면 6 − 답, 1 ~ 7이면 8 − 답). 아래 코드는 눈금이 1부터 시작한다고 보고 `MAX + 1`에서 뺀다. 0부터 시작하는 눈금이라면 `MAX`에서 빼야 한다. 원래 답 `raw`는 그대로 두고, 뒤집은 답을 `d`에 따로 만든다. 3절에서 뒤집기 전과 뒤를 비교하려면 둘 다 있어야 한다.

척도 점수는 문항 10개의 평균이다. `mean(axis=1)`은 사람마다(행마다) 평균을 낸다. 외향성 점수를 뒤집은 답으로 낸 것과 뒤집지 않은 답으로 낸 것의 분포를 나란히 그려 그림 2-2를 확인한다. 뒤집지 않으면 점수가 3 근처로 몰린다. 마지막 줄은 외향성 10문항에 모두 같은 숫자로 답한 사람의 수다(`nunique(axis=1)`은 사람마다 서로 다른 답의 가짓수).

In [ ]:
print(raw.groupby('E1')[['E3', 'E2']].mean().round(2))      # 표 2-1

d = raw.copy()
d[REV] = MAX + 1 - d[REV]                                   # 역문항의 답 뒤집기: 6 − 답

score = d[cols('E')].mean(axis=1)                           # 외향성 척도 점수(뒤집은 답)
naive = raw[cols('E')].mean(axis=1)                         # 뒤집지 않고 평균하면
print(f"2.5 ~ 3.5 안에 든 사람: 뒤집은 점수 {score.between(2.5, 3.5).mean():.0%}, 뒤집지 않은 점수 {naive.between(2.5, 3.5).mean():.0%}")

fig, axs = plt.subplots(1, 2, figsize=(8.4, 2.9), sharey=True)
bins = np.arange(0.95, 5.1, 0.1)
for ax, s, title, col in [(axs[0], score, '역문항의 답을 뒤집은 뒤', '#2a78d6'), (axs[1], naive, '뒤집지 않고 평균하면', '#eb6834')]:
    ax.hist(s, bins=bins, color=col)
    ax.set_title(title, loc='left')
    ax.set_xlabel('외향성 점수(문항 10개의 평균)')
axs[0].set_ylabel('사람 수')
plt.show()

print('외향성 10문항에 모두 같은 답:', (raw[cols('E')].nunique(axis=1) == 1).sum(), '명')

## 3. 크론바흐 α — 보고서의 표

크론바흐 α를 함수로 만든다. 3절의 참고 상자의 식 그대로다.

$$\alpha = \frac{k}{k-1}\left(1 - \frac{\text{문항 분산의 합}}{\text{합의 분산}}\right)$$

문항들이 함께 움직이면 합이 넓게 퍼져(합의 분산이 커서) α가 1에 가까워지고, 따로 움직이면 0에 가까워지며, 거꾸로 움직이면 음수가 된다. `x.var()`는 문항마다의 분산, `x.sum(axis=1).var()`는 사람마다 더한 합의 분산이다.

먼저 3절에 나온 상관계수(E1과 E3, E1과 O8, 외향성 문항끼리의 평균)를 확인하고, 두 문항만으로 α도 계산해 본다(함께 움직이는 E1 + E3, 따로 움직이는 E1 + O8). 거꾸로 움직이는 두 문항은 α가 음수가 되는 것도 본다(E1과, 답을 뒤집지 않은 E2). 그다음 다섯 척도마다 보고서의 표(문항 수, 역문항 수, α, 척도 점수의 평균과 표준편차)를 만들고, 뒤집기 전의 α(표 3-2)를 한 열로 덧붙인다. 신경성은 뒤집지 않아도 0.7을 넘는 것을 확인한다. 기준만 보면 실수를 놓친다.

In [ ]:
def alpha(x):
    k = x.shape[1]                                          # 문항 수
    return k / (k - 1) * (1 - x.var().sum() / x.sum(axis=1).var())

r = d[cols('E')].corr().values                             # 외향성 문항끼리의 상관계수 45개
print('상관계수 E1·E3:', round(raw['E1'].corr(raw['E3']), 2), ' E1·O8:', round(raw['E1'].corr(raw['O8']), 2),
      ' 외향성 문항끼리 평균:', round(r[np.triu_indices(10, 1)].mean(), 2))
print('α  E1 + E3:', round(alpha(raw[['E1', 'E3']]), 2),
      ' E1 + O8:', round(alpha(raw[['E1', 'O8']]), 2),
      ' E1 + E2(답을 뒤집지 않으면):', round(alpha(raw[['E1', 'E2']]), 2))

rows = []
for t, name in TRAITS.items():
    s = d[cols(t)].mean(axis=1)                             # 척도 점수
    rows.append([name, 10, sum(c in REV for c in cols(t)), alpha(d[cols(t)]), s.mean(), s.std(),
                 alpha(raw[cols(t)])])
table = pd.DataFrame(rows, columns=['척도', '문항', '역문항', 'α', '평균', '표준편차', '뒤집기 전 α'])
table.round(2)

### 문항 수와 α — 그림 3-3

척도마다 문항 10개 가운데 $k$개를 무작위로 뽑아 α를 계산하기를 300번 하고, 그 평균을 문항 수마다 그린다. `rng.choice(cols(t), k, replace=False)`가 문항 $k$개를 겹치지 않게 뽑는다. 시드(`default_rng(0)`)를 고정했으므로 몇 번을 실행해도 같은 문항이 뽑히고, 본문과 같은 숫자가 나온다. 뽑은 문항의 목록은 `draws`에 저장해 5절에서 다시 쓴다. 실행에 몇십 초 걸린다.

문항이 많을수록 α가 오르는 것을 확인한다. 문항마다의 운이 서로 상쇄되기 때문이다(3절).

In [ ]:
rng = np.random.default_rng(0)
K = range(2, 11)
draws = {t: {k: [list(rng.choice(cols(t), k, replace=False)) for _ in range(300)] for k in range(2, 10)}
         for t in TRAITS}                                   # 성격 × 문항 수마다 300번 뽑은 문항 목록
curve = {t: [np.mean([alpha(d[c]) for c in draws[t][k]]) if k < 10 else alpha(d[cols(t)]) for k in K]
         for t in TRAITS}
print(pd.DataFrame(curve, index=list(K)).rename(columns=TRAITS).round(2))

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.axhline(0.7, color='lightgray', ls='--')
for t in TRAITS:
    ax.plot(list(K), curve[t], marker='o', label=TRAITS[t])
ax.set_xlabel('척도에 넣은 문항 수')
ax.set_ylabel('크론바흐 α')
ax.legend(frameon=False)
plt.show()

### 문항 하나를 빼면 — 표 3-3

척도마다 문항을 하나씩 빼고 나머지 9문항의 α를 계산한다. 빼면 α가 오르는 문항만 출력한다. 네 문항이 나오고, 오르는 폭은 0.001 ~ 0.006이다. 이 정도 차이라면 문항을 뺄 까닭이 없다(3절).

In [ ]:
for t in TRAITS:
    a10 = alpha(d[cols(t)])
    for c in cols(t):
        a9 = alpha(d[[x for x in cols(t) if x != c]])     # c를 뺀 9문항
        if a9 > a10:
            print(f"{TRAITS[t]} {c} {ITEM_KO[c]}  10문항 {a10:.3f} → 뺀 9문항 {a9:.3f}")

## 4. 요인분석 — 보고서의 그림

50문항으로 탐색적 요인분석을 한다. 문항을 표준화(평균을 빼고 표준편차로 나누기)한 뒤, 요인 5개, 베리맥스 회전으로 `FactorAnalysis`를 맞춘다. `components_`에 요인 × 문항의 적재값이 들어 있으므로, `.T`로 뒤집어 문항 × 요인의 표 `L`로 만든다.

요인분석은 요인을 어떤 순서로 내놓을지, 부호를 어느 쪽으로 할지 정하지 않는다. 요인 1이 외향성이라는 보장도, 외향성 문항의 적재값이 양수라는 보장도 없다. 그래서 성격마다 자기 문항 10개가 가장 크게 적재된 요인을 찾아 성격 순서로 늘어놓고, 자기 문항의 적재값이 양수가 되도록 부호를 바꾼다. 요인의 뜻은 바뀌지 않는다. 이렇게 맞춘 뒤의 열 이름이 성격의 앞 글자다.

보고서의 그림은 두 패널이다. 왼쪽은 문항 50개끼리의 상관계수(`d[ITEMS].corr()`)를 열지도(`imshow`)로 그린 것이다. 오른쪽은 문항마다 적재값이 가장 큰 요인을 그 문항의 묶음으로 삼고(`idxmax`), 묶음마다 어느 성격의 문항이 몇 개 들었는지를 센 표(`pd.crosstab`)다. 이어서 보고서의 그림 주에 적힌 숫자와 4절의 표 4-2를 계산한다 — 같은 성격의 문항끼리와 다른 성격의 문항끼리의 상관계수 평균, 자기 묶음의 적재값 범위. 4절의 "요인분석 결과를 더 자세히 볼 때"에 나오는 다른 요인 적재값 가운데 가장 큰 것도 계산한다. 책에는 싣지 않았지만 요인분석 결과에 흔히 함께 나오는 다섯 요인이 설명한 분산(적재값 제곱의 합 ÷ 문항 수)도 계산해 둔다.

In [ ]:
zs = (d[ITEMS] - d[ITEMS].mean()) / d[ITEMS].std()        # 표준화

def factor(rotation):
    fa = FactorAnalysis(n_components=5, rotation=rotation, random_state=0).fit(zs)
    return pd.DataFrame(fa.components_.T, index=ITEMS)     # 문항 × 요인의 적재값

L = factor('varimax')
order = []
for t in TRAITS:                                           # 성격마다 자기 문항이 가장 크게 적재된 요인
    order.append(L.loc[cols(t)].abs().mean().drop(order).idxmax())
L = L[order]
L.columns = list(TRAITS)
for t in TRAITS:                                           # 자기 문항의 적재값이 양수가 되게
    if L.loc[cols(t), t].mean() < 0:
        L[t] = -L[t]

c = d[ITEMS].corr()                                        # 문항 50개끼리의 상관계수
group = L.abs().idxmax(axis=1)                             # 문항마다 적재값이 가장 큰 요인 = 그 문항의 묶음
count = pd.crosstab(group, [i[0] for i in ITEMS]).reindex(index=list(TRAITS), columns=list(TRAITS)).fillna(0).astype(int)
count.index = [f'묶음 {k + 1}' for k in range(5)]
count.columns = list(TRAITS.values())

fig, axs = plt.subplots(1, 2, figsize=(8.4, 3.0))
v = c.values.copy()
np.fill_diagonal(v, np.nan)                                # 자기 자신과의 상관계수 1은 비운다
axs[0].imshow(v, cmap='Blues', vmin=0, vmax=0.7)
axs[0].set_xticks([4.5 + 10 * k for k in range(5)], list(TRAITS.values()))   # 성격마다 문항 10개
axs[0].set_yticks([4.5 + 10 * k for k in range(5)], list(TRAITS.values()))
axs[0].set_title('문항 50개끼리의 상관계수')
axs[1].axis('off')
axs[1].table(cellText=count.values, rowLabels=count.index, colLabels=count.columns, loc='center')
axs[1].set_title('요인분석이 찾은 묶음에 든 문항 수')
plt.show()

same = np.array([[a[0] == b[0] for b in ITEMS] for a in ITEMS])      # 같은 성격의 두 문항인가
iu = np.triu_indices(50, 1)                                          # 서로 다른 두 문항의 짝 1,225개
print(f'상관계수 평균: 같은 성격의 문항끼리 {c.values[iu][same[iu]].mean():.2f}, 다른 성격의 문항끼리 {c.values[iu][~same[iu]].mean():.2f}')
mean = pd.DataFrame({TRAITS[b]: [c.loc[cols(a), cols(b)].values[np.triu_indices(10, 1)].mean() if a == b
                                 else c.loc[cols(a), cols(b)].values.mean() for a in TRAITS] for b in TRAITS},
                    index=list(TRAITS.values()))
print(mean.round(2))                                       # 표 4-2

own = pd.Series({c: L.loc[c, c[0]] for c in ITEMS})                  # 자기 요인 적재값
cross = pd.Series({c: L.loc[c].drop(c[0]).abs().max() for c in ITEMS})  # 다른 요인 가운데 가장 큰 것
print('자기 요인에 가장 크게 적재된 문항:', (L.abs().idxmax(axis=1) == [c[0] for c in ITEMS]).sum(), '/ 50')
print(f'자기 요인 적재값 {own.min():.2f} ~ {own.max():.2f}, 0.4 미만', own[own < 0.4].round(2).to_dict())
print(f'다른 요인 적재값 최대 {cross.max():.2f} ({cross.idxmax()} {ITEM_KO[cross.idxmax()]})')
print(f'다섯 요인이 설명한 분산 {(L ** 2).sum().sum() / 50:.0%}')

### 문항 몇 개의 적재값 — 표 4-3

4절의 표 4-3에 나온 여덟 문항의 적재값이다. 마지막 열은 문항마다 다섯 요인이 함께 설명한 몫(적재값 제곱의 합, 공통성)이다. C4가 자기 요인(C)에 0.55로 적재되면서 신경성(N)에도 −0.37로 적재되는 교차 적재를 확인한다. 공통성이 가장 큰 문항과 가장 작은 문항도 출력한다.

In [ ]:
show = ['E1', 'E7', 'N8', 'N4', 'A4', 'A3', 'C4', 'O9']
comm = (L ** 2).sum(axis=1)                                # 공통성
print(L.loc[show].assign(공통성=comm[show]).round(2))
print(f'공통성 가장 큰 문항 {comm.idxmax()} {comm.max():.0%}, 가장 작은 문항 {comm.idxmin()} {comm.min():.0%}')

### 몇 개로 묶을까 — 고유값(그림 4-4)

요인 수를 모를 때 먼저 보는 고유값이다. 문항끼리 함께 움직이는 정도를 모은 표(상관행렬, `zs.corr()`)의 고유값을 `np.linalg.eigvalsh`로 구하고 큰 순서로 늘어놓는다. 고유값을 모두 더하면 문항 수 50이 된다. 고유값 1을 넘는 요인은 8개이고, 그래프는 다섯째 요인 뒤에서 꺾인다(4절).

In [ ]:
ev = np.sort(np.linalg.eigvalsh(zs.corr()))[::-1]          # 큰 순서
print('고유값 앞 10개', ev[:10].round(2), ' 1을 넘는 것', (ev > 1).sum(), '개, 합', round(ev.sum(), 1))

fig, ax = plt.subplots(figsize=(6.4, 2.9))
ax.plot(range(1, 16), ev[:15], marker='o', color='#2a78d6')
ax.axhline(1, color='lightgray', ls='--')
ax.set_xticks(range(1, 16))
ax.set_xlabel('요인 번호(큰 순서)')
ax.set_ylabel('고유값')
plt.show()

### 더 해 보기 — 회전하기 전과 비교

회전하지 않은 요인분석(`rotation=None`)과 비교한다. 회전하기 전에는 요인 1에 여러 성격의 문항이 한꺼번에 적재되어, 문항이 성격별로 나뉘어 보이지 않는다. 회전 전 요인 1에 적재값 0.3 이상으로 적재된 문항이 29개인 것을 확인한다. 회전은 요인의 축만 돌리므로, 다섯 요인이 함께 설명한 분산은 회전 전과 뒤가 같다. 책 본문은 회전을 이름과 뜻만 소개한다.

In [ ]:
U = factor(None)                                           # 회전하기 전
print('회전 전 요인 1에 0.3 이상 적재된 문항:', (U[0].abs() >= 0.3).sum(), '개')
print(f'설명한 분산: 회전 전 {(U ** 2).sum().sum() / 50:.3f}, 회전 뒤 {(L ** 2).sum().sum() / 50:.3f}')

## 5. 감쇠 — 문항 하나로 측정하면

친화성 척도 점수를 종속변수로, 외향성 점수를 설명변수로 회귀분석한다. 외향성을 E1 한 문항으로 측정했을 때와 10문항의 척도 점수로 측정했을 때의 기울기를 비교한다(그림 5-1). 회귀식에 쓰기 쉽게 척도 점수를 `s`라는 표에 모은다.

측정이 거칠면 기울기가 0 쪽으로 작아지는 것을 확인한다(감쇠). 표 5-1은 문항 수를 늘려 가며 같은 계산을 한 것이다. 3절에서 뽑아 둔 외향성 문항 목록 `draws['E']`를 그대로 쓰므로, α도 3절의 숫자와 같다. 기울기를 1,000번 넘게 구하므로 회귀식 대신 기울기만 빠르게 구하는 `np.polyfit(x, y, 1)[0]`을 쓴다. 문항 하나일 때의 값은 외향성 문항 10개 하나하나의 기울기를 평균한 것이다(책의 표 5-1에는 2문항부터 실었다. 문항 하나는 그림 5-1의 E1으로 말한다).

In [ ]:
s = pd.DataFrame({t: d[cols(t)].mean(axis=1) for t in TRAITS})   # 척도 점수 5개
s['E1'] = d['E1']
b1 = smf.ols('A ~ E1', s).fit().params['E1']
b10 = smf.ols('A ~ E', s).fit().params['E']
print(f'친화성을 설명하는 기울기: E1 한 문항 {b1:.2f}, 외향성 10문항 {b10:.2f}')

slope = lambda x: np.polyfit(x, s['A'], 1)[0]
rows = [[1, np.nan, np.mean([slope(d[c]) for c in cols('E')])]]
for k in [2, 3, 5]:
    rows.append([k, np.mean([alpha(d[c]) for c in draws['E'][k]]),
                 np.mean([slope(d[c].mean(axis=1)) for c in draws['E'][k]])])
rows.append([10, alpha(d[cols('E')]), slope(s['E'])])
print(pd.DataFrame(rows, columns=['문항 수', 'α', '기울기']).round(2))

fig, axs = plt.subplots(1, 2, figsize=(8.4, 3.0), sharey=True)
for ax, x, g, title, col in [(axs[0], s['E1'], s['E1'], '외향성을 문항 하나(E1)로 측정', '#eb6834'),
                             (axs[1], s['E'], (s['E'] * 2).round() / 2, '외향성을 문항 10개로 측정', '#2a78d6')]:
    m = s['A'].groupby(g).agg(['mean', 'size'])
    m = m[m['size'] >= 100]                                # 100명 이상인 점수만 점으로
    ax.scatter(m.index, m['mean'], color=col)
    b, a = np.polyfit(x, s['A'], 1)
    ax.plot([1, 5], [a + b * 1, a + b * 5], color=col)
    ax.set_title(f'{title}, 기울기 {b:.2f}', loc='left')
    ax.set_xlabel('외향성 점수')
axs[0].set_ylabel('친화성 점수의 평균')
plt.show()

## 6. 더 해 보기: 채점 키를 자료에서 짐작하면

본문에서는 다루지 않은 계산이다. 채점 키가 없을 때 자료의 숫자로 역문항을 고를 수 있을까? 흔히 떠올리는 방법은 문항마다 나머지 9문항의 합(뒤집기 전의 원래 답)과 거꾸로 움직이는지 보는 것이다. 여기서는 나머지 합을 그 문항으로 회귀분석한 기울기가 음수인 문항을 역문항으로 골라 본다.

신경성, 친화성, 개방성은 채점 키와 똑같이 골라진다. 역문항이 적어서 나머지 합이 정문항 쪽으로 움직이기 때문이다. 그런데 정문항과 역문항이 5 대 5인 외향성에서는 나머지 합이 이쪽저쪽으로 상쇄되어, 정문항 E5, E7, E9까지 역문항으로 골라진다. 성실성에서도 정문항 C5가 섞인다. 그렇게 고른 키로 뒤집으면 외향성의 α가 0.89에서 0.03으로, 성실성은 0.81에서 0.66으로 떨어진다. 채점 키는 문항의 뜻에서 나와야 한다. 직접 해 보기 1의 프롬프트에 "채점 키를 자료에서 짐작하지 마"를 넣은 까닭이다.

In [ ]:
for t in TRAITS:
    rest = lambda c: raw[[x for x in cols(t) if x != c]].sum(axis=1)   # 나머지 9문항의 합
    guess = [c for c in cols(t) if np.polyfit(raw[c], rest(c), 1)[0] < 0]
    g = raw[cols(t)].copy()
    g[guess] = MAX + 1 - g[guess]
    print(f"{TRAITS[t]} 짐작한 역문항 {guess}")
    print(f"      채점 키 {[c for c in cols(t) if c in REV]},  α: 짐작한 키 {alpha(g):.2f}, 채점 키 {alpha(d[cols(t)]):.2f}")

## 7. 더 해 보기: 하만 단일요인검정

5절에서 이름과 비판을 소개한 하만 단일요인검정이다. 모든 척도의 문항을 한꺼번에 넣고 회전하지 않은 분석을 해서, 첫 요인 하나가 문항들의 흩어짐을 얼마나 설명하는지 본다. 첫 요인이 절반 넘게 설명하면 공통방법편향을 의심한다는 기준이 흔히 쓰인다. 4절의 고유값으로 계산할 수 있다. 첫 고유값을 고유값의 합(문항 수)으로 나눈 것이다.

이 자료에서는 첫 요인이 약 16%를 설명해 기준 50%에 한참 못 미친다. 다만 5절에서 본 것처럼, 이 검정을 통과했다고 공통방법편향이 없다는 증거가 되지는 않는다.

In [ ]:
print(f'첫 요인이 설명한 몫 {ev[0] / ev.sum():.1%}')

## 정리

이 노트북의 코드는 이 장에서 본 숫자를 직접 만들어 냈다. 역문항의 답을 뒤집고 평균을 낸 것이 척도 점수였고(2절), 크론바흐 α는 문항 분산의 합과 합의 분산을 비교한 것이었다(3절). 요인분석은 `FactorAnalysis` 한 줄이지만, 요인의 순서와 부호를 맞추고 문항을 읽어 이름을 붙이는 일은 사람이 했다(4절). 척도 점수를 회귀식에 넣으면, 문항 하나로 측정했을 때보다 관계가 제 크기에 가깝게 나왔다(5절).

자기 설문으로 분석하려면 1절에서 자기 파일을 읽고(쉼표로 나뉜 파일이면 `sep='\t'`를 지운다) `TRAITS`(척도)와 `ITEMS`(문항 이름)를 바꾸고, 2절에서 `cols`(척도마다의 문항 이름), `REV`(채점 키), `MAX`(눈금의 가장 큰 값)를 바꾼다. 빠진 답을 어떻게 적었는지는 1절에서 먼저 확인한다. 다만 이 노트북에는 이 장의 자료에 맞춰 쓴 곳이 남아 있다. 척도마다 문항이 10개, 모두 50개이고 문항 이름의 첫 글자가 척도라고 보고 계산하는 곳(`range(1, 11)`, `10`, `50`, `c[0]`)이 있고, 5절의 회귀분석은 외향성(`'E'`, `'E1'`)과 친화성(`'A'`)을 직접 적었다. 자기 자료가 이와 다르면 이런 곳도 함께 고치고, 4절의 `n_components`를 척도의 수로 바꾼다. 어디를 고칠지는 이 노트북과 자기 자료의 변수 목록을 AI에게 보여 주고 물어봐도 된다. AI가 만든 보고서를 받았다면, 이 코드로 같은 숫자가 나오는지 확인해 보는 것이 가장 확실한 점검이다.